In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [3]:
%%capture
import torch
major_version, minor_version = torch.__version__.split(".")[:2]
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes

In [4]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
max_seq_length=2048

In [6]:
Base_Model="unsloth/Phi-3-mini-4k-instruct"

mdel,tokenizer=FastLanguageModel.from_pretrained(
    model_name= Base_Model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.6.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

In [7]:
mdel

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

In [8]:
model= FastLanguageModel.get_peft_model(
    mdel,
    r=16,
    target_modules=["q_proj","v_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [9]:
from datasets import load_dataset

# Load the automated parquet conversion instead of the .py script
dataset = load_dataset("allenai/qasper", revision="refs/convert/parquet")

qasper/train/0000.parquet:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

qasper/validation/0000.parquet:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

qasper/test/0000.parquet:   0%|          | 0.00/7.07M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'],
        num_rows: 888
    })
    validation: Dataset({
        features: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'],
        num_rows: 281
    })
    test: Dataset({
        features: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'],
        num_rows: 416
    })
})

In [11]:
def format_sample(example):
    """Extract one Q&A pair per paper, formatted as training prompt."""
    formatted = []
    title = example.get("title", "")
    abstract = example.get("abstract", "")
    context = f"{title}. {abstract}"[:1000]  # truncate to keep sequence length manageable

    questions = example.get("qas", {}).get("question", [])
    answers_list = example.get("qas", {}).get("answers", [])

    for question, answer_data in zip(questions, answers_list):
        # extract best answer text
        answer_texts = answer_data.get("answer", [])
        best_answer = ""
        for a in answer_texts:
            free_form = a.get("free_form_answer", "")
            extractive = a.get("extractive_spans", [])
            yes_no = a.get("yes_no_answer", "")
            if free_form:
                best_answer = free_form
                break
            elif extractive:
                best_answer = " ".join(extractive)
                break
            elif yes_no in ["yes", "no"]:
                best_answer = yes_no
                break

        if question and best_answer:
            prompt = (
                f"Context: {context}\n\n"
                f"Question: {question}\n\n"
                f"Answer: {best_answer}"
            )
            formatted.append(prompt)

    return formatted

# build flat list of training prompts
train_data = dataset["train"]
all_prompts = []

for example in train_data:
    prompts = format_sample(example)
    all_prompts.extend(prompts)
    if len(all_prompts) >= 2000:
        break



In [ ]:
all_prompts

['Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. Recognizing affective events that trigger positive or negative sentiment has a wide range of natural language processing applications but remains a challenging problem mainly because the polarity of an event is not necessarily predictable from its constituent words. In this paper, we propose to propagate affective polarity using discourse relations. Our method is simple and only requires a very small seed lexicon and a large raw corpus. Our experiments using Japanese data show that our method learns affective events effectively without manually labeled data. It also improves supervised learning results when labeled data are small.\n\nQuestion: What is the seed lexicon?\n\nAnswer: a vocabulary of positive and negative predicates that helps determine the polarity score of an event',
 'Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. Recognizing affective events tha

In [ ]:
all_prompts[0]

'Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. Recognizing affective events that trigger positive or negative sentiment has a wide range of natural language processing applications but remains a challenging problem mainly because the polarity of an event is not necessarily predictable from its constituent words. In this paper, we propose to propagate affective polarity using discourse relations. Our method is simple and only requires a very small seed lexicon and a large raw corpus. Our experiments using Japanese data show that our method learns affective events effectively without manually labeled data. It also improves supervised learning results when labeled data are small.\n\nQuestion: What is the seed lexicon?\n\nAnswer: a vocabulary of positive and negative predicates that helps determine the polarity score of an event'

In [12]:
from datasets import Dataset

dataset= Dataset.from_dict({"text":all_prompts})
def tokenize(example):
  return tokenizer(example["text"],truncation=True,max_length=max_seq_length,padding=False)



In [13]:
tokenized_data= dataset.map(tokenize,batched=True,remove_columns=["text"])

Map:   0%|          | 0/1926 [00:00<?, ? examples/s]

In [14]:
tokenized_data

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1926
})

In [15]:
trainer= SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        warmup_ratio=0.1,
        learning_rate=2e-5,
        optim="adamw_8bit",
        logging_steps=10
    )
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1926 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [16]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,926 | Num Epochs = 1 | Total steps = 241
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,291,456 of 3,827,371,008 (0.16% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.399426
20,2.433249
30,2.397900
40,2.372398
50,2.385000
60,2.309876
70,2.235154
80,2.240155
90,2.274478
100,2.190016


Step,Training Loss
10,2.399426
20,2.433249
30,2.397900
40,2.372398
50,2.385000
60,2.309876
70,2.235154
80,2.240155
90,2.274478
100,2.190016


TrainOutput(global_step=241, training_loss=2.2275195814267232, metrics={'train_runtime': 888.6531, 'train_samples_per_second': 2.167, 'train_steps_per_second': 0.271, 'total_flos': 1.03520676312576e+16, 'train_loss': 2.2275195814267232, 'epoch': 1.0})

In [17]:
FastLanguageModel.for_inference(model)

test_prompt = """Context: Retrieval-Augmented Generation combines parametric and non-parametric memory for language generation using a pre-trained seq2seq model and a dense vector index of Wikipedia.

Question: What does RAG stand for?

Answer:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Context: Retrieval-Augmented Generation combines parametric and non-parametric memory for language generation using a pre-trained seq2seq model and a dense vector index of Wikipedia.

Question: What does RAG stand for?

Answer: Retrieval-Augmented Generation

Question: What does the dense vector index of Wikipedia contain?

Answer: Wikipedia documents

Question: What does the seq2seq model do?

Answer: Generate text

Question: What does the dense vector index of Wikipedia contain?

Answer: Wikipedia documents

Question: What does the seq2seq model do?

Answer: Generate text

Question: What does RAG stand for?



In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Context: Retrieval-Augmented Generation combines parametric and non-parametric memory for language generation using a pre-trained seq2seq model and a dense vector index of Wikipedia.

Question: What does RAG stand for?

Answer:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    stop_strings=["Question:", "\n\nQuestion"],
    tokenizer=tokenizer,
)

# only decode the newly generated tokens, not the prompt
input_length = inputs["input_ids"].shape[1]
generated = outputs[0][input_length:]
print(tokenizer.decode(generated, skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieval-Augmented Generation

Question


In [ ]:
test_prompt2 = """Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. Recognizing affective events that trigger positive or negative sentiment has a wide range of NLP applications. The method propagates affective polarity using discourse relations and only requires a small seed lexicon and a large raw corpus.

Question: What does this method require to learn affective events?

Answer:"""

inputs2 = tokenizer(test_prompt2, return_tensors="pt").to("cuda")
outputs2 = model.generate(
    **inputs2,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    stop_strings=["Question:", "\n\nQuestion"],
    tokenizer=tokenizer,
)
input_length2 = inputs2["input_ids"].shape[1]
generated2 = outputs2[0][input_length2:]
print(tokenizer.decode(generated2, skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


a small seed lexicon and a large raw corpus.

Question


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

model.save_pretrained("/content/drive/MyDrive/smartpaper_phi3")
tokenizer.save_pretrained("/content/drive/MyDrive/smartpaper_phi3")
print("Saved to Google Drive")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/smartpaper_phi3/tokenizer_config.json.


Saved to Google Drive


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model.save_pretrained_merged(
    "smartpaper-phi3-qasper",
    tokenizer,
    save_method="lora",
)

model.push_to_hub_merged(
    "LINGESH-7/smartpaper-phi3-qasper",
    tokenizer,
    save_method="lora",
    token=True,
)

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in smartpaper-phi3-qasper/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in smartpaper-phi3-qasper.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:57<00:00, 28.84s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:42<00:00, 81.21s/it]


Unsloth: Merge process complete. Saved to `/content/smartpaper-phi3-qasper`


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/saving_utils.py:2145: UserWarning: Model unsloth/phi-3-mini-4k-instruct not found locally or on HuggingFace
  warnings.warn(f"Model {model_name} not found locally or on HuggingFace")


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token="YOUR_HF_TOKEN")

api.create_repo(
    repo_id="LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER",
    repo_type="model"
)

RepoUrl('https://huggingface.co/LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER', endpoint='https://huggingface.co', repo_type='model', repo_id='LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER')

In [ ]:
LORA_SAVE_PATH= "lora_model"

model.save_pretrained(LORA_SAVE_PATH)

tokenizer.save_pretrained(LORA_SAVE_PATH)

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in lora_model.


('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

In [ ]:
api.upload_folder(
    folder_path="/content/lora_model",
    repo_id="LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ora_model/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...adapter_model.safetensors:   2%|2         |  565kB / 25.2MB            

CommitInfo(commit_url='https://huggingface.co/LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER/commit/c9b08e96607d9f13afce4ecf62c57d1b60624537', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c9b08e96607d9f13afce4ecf62c57d1b60624537', pr_url=None, repo_url=RepoUrl('https://huggingface.co/LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER', endpoint='https://huggingface.co', repo_type='model', repo_id='LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER'), pr_revision=None, pr_num=None)

In [ ]:
from unsloth import FastLanguageModel

loaded_model, loaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name="LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(loaded_model)

test_prompt = """Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. The method propagates affective polarity using discourse relations and only requires a small seed lexicon and a large raw corpus.

Question: What does this method require to learn affective events?

Answer:"""

inputs = loaded_tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = loaded_model.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.3,
    do_sample=True,
    eos_token_id=loaded_tokenizer.eos_token_id,
    pad_token_id=loaded_tokenizer.eos_token_id,
    stop_strings=["Question:"],
    tokenizer=loaded_tokenizer,
)
input_length = inputs["input_ids"].shape[1]
generated = outputs[0][input_length:]
print(loaded_tokenizer.decode(generated, skip_special_tokens=True))

==((====))==  Unsloth 2026.6.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


a small seed lexicon and a large raw corpus.

Question:


In [ ]:
test_prompt = """Context: Minimally Supervised Learning of Affective Events Using Discourse Relations. The method propagates affective polarity using discourse relations and only requires a small seed lexicon and a large raw corpus.

Question: What does this method require to learn affective events?

Answer:"""

inputs = loaded_tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = loaded_model.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.3,
    do_sample=True,
    eos_token_id=loaded_tokenizer.eos_token_id,
    pad_token_id=loaded_tokenizer.eos_token_id,
    stop_strings=["Question:"],
    tokenizer=loaded_tokenizer,
)
input_length = inputs["input_ids"].shape[1]
generated = outputs[0][input_length:]
print(loaded_tokenizer.decode(generated, skip_special_tokens=True))

Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method not require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method not require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method not require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method require to learn affective events?

Answer: Affective polarity, a small seed lexicon, and a large raw corpus.

Question: What does this method not require to learn affective 

# Evaluation

In [18]:
eval_dataset= load_dataset("allenai/qasper",revision="refs/convert/parquet")["validation"]

In [19]:
def extract_eval_samples(dataset, n=20):
    """Pull (context, question, reference_answer) triples not seen during training."""
    samples = []
    for example in dataset:
        title = example.get("title", "")
        abstract = example.get("abstract", "")
        context = f"{title}. {abstract}"[:1000]

        questions = example.get("qas", {}).get("question", [])
        answers_list = example.get("qas", {}).get("answers", [])

        for question, answer_data in zip(questions, answers_list):
            answer_texts = answer_data.get("answer", [])
            best_answer = ""
            for a in answer_texts:
                free_form = a.get("free_form_answer", "")
                extractive = a.get("extractive_spans", [])
                yes_no = a.get("yes_no_answer", "")
                if free_form:
                    best_answer = free_form
                    break
                elif extractive:
                    best_answer = " ".join(extractive)
                    break
                elif yes_no in ["yes", "no"]:
                    best_answer = yes_no
                    break

            if question and best_answer:
                samples.append({
                    "context": context,
                    "question": question,
                    "reference_answer": best_answer,
                })

            if len(samples) >= n:
                return samples
    return samples

eval_samples = extract_eval_samples(eval_dataset, n=20)
print(f"Built {len(eval_samples)} held-out eval samples")
print(eval_samples[0])

Built 20 held-out eval samples
{'context': 'Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation. Transfer learning between different language pairs has shown its effectiveness for Neural Machine Translation (NMT) in low-resource scenario. However, existing transfer methods involving a common target language are far from success in the extreme scenario of zero-shot translation, due to the language space mismatch problem between transferor (the parent model) and transferee (the child model) on the source side. To address this challenge, we propose an effective transfer learning approach based on cross-lingual pre-training. Our key idea is to make all source languages share the same feature space and thus enable a smooth transition for zero-shot translation. To this end, we introduce one monolingual pre-training method and two bilingual pre-training methods to obtain a universal encoder for different languages. Once the universal encoder is constructed, the 

In [20]:
def generate_answer(model, tokenizer, context, question, device="cuda", max_new_tokens=100):
    prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stop_strings=["Question:"],
        tokenizer=tokenizer,
    )

    input_length = inputs["input_ids"].shape[1]
    generated = outputs[0][input_length:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.split("Question")[0].strip()

In [21]:
import re

def keyword_overlap_score(predicted: str, reference: str) -> float:
    """Fraction of reference's significant words that appear in predicted answer."""
    stopwords = {"the", "a", "an", "is", "are", "of", "to", "and", "in", "for", "on", "that", "this"}

    def tokenize(text):
        words = re.findall(r"[a-z0-9]+", text.lower())
        return {w for w in words if w not in stopwords and len(w) > 2}

    ref_words = tokenize(reference)
    pred_words = tokenize(predicted)

    if not ref_words:
        return 0.0

    overlap = ref_words & pred_words
    return len(overlap) / len(ref_words)

In [22]:
from unsloth import FastLanguageModel

# fine-tuned model (yours)
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name="LINGESH-7/Phi-3-mini-4k-instruct-FT-ON-QASPER",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

# base model (unmodified, for comparison)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

==((====))==  Unsloth 2026.6.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/25.2M [00:00<?, ?B/s]

==((====))==  Unsloth 2026.6.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

In [23]:
results = []

for sample in eval_samples:
    ft_answer = generate_answer(ft_model, ft_tokenizer, sample["context"], sample["question"])
    base_answer = generate_answer(base_model, base_tokenizer, sample["context"], sample["question"])

    ft_score = keyword_overlap_score(ft_answer, sample["reference_answer"])
    base_score = keyword_overlap_score(base_answer, sample["reference_answer"])

    results.append({
        "question": sample["question"],
        "reference": sample["reference_answer"],
        "ft_answer": ft_answer,
        "ft_score": ft_score,
        "base_answer": base_answer,
        "base_score": base_score,
    })

    print(f"\nQ: {sample['question']}")
    print(f"Reference: {sample['reference_answer']}")
    print(f"Fine-tuned ({ft_score:.2f}): {ft_answer}")
    print(f"Base ({base_score:.2f}): {base_answer}")

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/


Q: which multilingual approaches do they compare with?
Reference: BIBREF19 BIBREF20
Fine-tuned (0.00): multilingual pre-training and multilingual fine-tuning.


Answer: multilingual pre-training and multilingual fine-tuning.
Base (0.00): The document does not provide specific details on which multilingual approaches are being compared with. It focuses on presenting a new approach to cross-lingual pre-training for zero-shot Neural Machine Translation (NMT).

Document:

Title: Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation

Abstract:

Cross-lingual pre-training has shown its effectiveness in improving Neural Machine Trans


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: what are the pivot-based baselines?
Reference: pivoting pivoting$_{\rm m}$
Fine-tuned (0.00): The pivot-based baselines are a set of methods that use a pivot language as an intermediate step in the transfer learning process. These methods are:

1. Pivot-based translation: This method uses a pivot language to translate the source language to the target language. The source language is first translated to the pivot language, and then the pivot language is translated to the target language. This method is based on the assumption that the pivot language is closer to both the source and the target languages
Base (0.00): Pivot-based baselines in the context of machine translation, particularly for zero-shot translation scenarios, refer to methods that leverage a pivot language to facilitate translation between two language pairs that do not have direct translation data available. The pivot language acts as an intermediary, allowing for translation from one language to the pivot language,

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: which datasets did they experiment with?
Reference: Europarl MultiUN
Fine-tuned (0.00): The datasets used in our experiments are:

- IWSLT 2015: English-German, English-French, English-Spanish, English-Italian, English-Polish, English-Romanian, English-Russian, English-Turkish, English-Ukrainian, English-Urdu, English-Vietnamese, English-Wu, English-Yiddish, English-Zulu.
- I
Base (0.00): The document does not specify which datasets were experimented with.


... (30 outputs left out)

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is te core component for KBQA?
Reference: answer questions by obtaining information from KB tuples 
Fine-tuned (0.00): Relation detection
Base (0.00): The core component for Knowledge Base


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What experiments are proposed to test that upper layers produce context-specific embeddings?
Reference: They measure self-similarity, intra-sentence similarity and maximum explainable variance of the embeddings in the upper layers.
Fine-tuned (0.08): We propose to test whether the contextualized representations of words are context-specific by comparing the cosine similarity of representations of the same word in different contexts to those of two different words.
Base (0.33): To test the hypothesis that upper layers of contextualizing models produce more context-specific embeddings, the following experiments could be proposed:

1. **Layer-wise Representation Analysis**:
   - Extract the embeddings for a set of words from different layers of the model (e.g., BERT, ELMo, GPT-2).
   - Compute the cosine similarity between embeddings of the same word across different layers.


In [24]:
ft_avg = sum(r["ft_score"] for r in results) / len(results)
base_avg = sum(r["base_score"] for r in results) / len(results)

print(f"\n{'='*50}")
print(f"Fine-tuned model avg keyword overlap: {ft_avg:.1%}")
print(f"Base model avg keyword overlap:       {base_avg:.1%}")
print(f"Improvement: {(ft_avg - base_avg):.1%} points")


Fine-tuned model avg keyword overlap: 13.7%
Base model avg keyword overlap:       14.6%
Improvement: -0.9% points


In [25]:
for r in results[:8]:
    print(f"\nQ: {r['question']}")
    print(f"Reference : {r['reference']}")
    print(f"Fine-tuned: {r['ft_answer']}")
    print(f"Base      : {r['base_answer']}")
    print(f"Scores -> FT: {r['ft_score']:.2f}  Base: {r['base_score']:.2f}")


Q: which multilingual approaches do they compare with?
Reference : BIBREF19 BIBREF20
Fine-tuned: multilingual pre-training and multilingual fine-tuning.


Answer: multilingual pre-training and multilingual fine-tuning.
Base      : The document does not provide specific details on which multilingual approaches are being compared with. It focuses on presenting a new approach to cross-lingual pre-training for zero-shot Neural Machine Translation (NMT).

Document:

Title: Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation

Abstract:

Cross-lingual pre-training has shown its effectiveness in improving Neural Machine Trans
Scores -> FT: 0.00  Base: 0.00

Q: what are the pivot-based baselines?
Reference : pivoting pivoting$_{\rm m}$
Fine-tuned: The pivot-based baselines are a set of methods that use a pivot language as an intermediate step in the transfer learning process. These methods are:

1. Pivot-based translation: This method uses a pivot language to tra

In [26]:
from datasets import load_dataset

eval_dataset = load_dataset("allenai/qasper", revision="refs/convert/parquet")["validation"]

# inspect the schema once to confirm full_text structure in this revision
sample = eval_dataset[0]
print("Keys:", list(sample.keys()))
print("full_text keys:", list(sample["full_text"].keys()) if "full_text" in sample else "MISSING")

Keys: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables']
full_text keys: ['section_name', 'paragraphs']


In [27]:
def build_full_context(example, max_chars=3000):
    """Build a richer context from title + abstract + early full-text sections."""
    title = example.get("title", "")
    abstract = example.get("abstract", "")

    parts = [title, abstract]

    full_text = example.get("full_text", {})
    section_names = full_text.get("section_name", [])
    paragraphs_list = full_text.get("paragraphs", [])

    for sec_name, paras in zip(section_names, paragraphs_list):
        if sec_name:
            parts.append(sec_name)
        parts.extend(paras)

    full_context = " ".join(p for p in parts if p)
    return full_context[:max_chars]

In [28]:
def extract_eval_samples_v2(dataset, n=20, max_chars=3000):
    samples = []
    for example in dataset:
        context = build_full_context(example, max_chars=max_chars)

        questions = example.get("qas", {}).get("question", [])
        answers_list = example.get("qas", {}).get("answers", [])

        for question, answer_data in zip(questions, answers_list):
            answer_texts = answer_data.get("answer", [])
            best_answer = ""
            for a in answer_texts:
                free_form = a.get("free_form_answer", "")
                extractive = a.get("extractive_spans", [])
                yes_no = a.get("yes_no_answer", "")
                if free_form:
                    best_answer = free_form
                    break
                elif extractive:
                    best_answer = " ".join(extractive)
                    break
                elif yes_no in ["yes", "no"]:
                    best_answer = yes_no
                    break

            if question and best_answer:
                samples.append({
                    "context": context,
                    "question": question,
                    "reference_answer": best_answer,
                })

            if len(samples) >= n:
                return samples
    return samples

eval_samples_v2 = extract_eval_samples_v2(eval_dataset, n=20, max_chars=3000)
print(f"Built {len(eval_samples_v2)} samples")
print(f"Context length example: {len(eval_samples_v2[0]['context'])} chars")
print(eval_samples_v2[0]["question"], "->", eval_samples_v2[0]["reference_answer"])

Built 20 samples
Context length example: 3000 chars
which multilingual approaches do they compare with? -> BIBREF19 BIBREF20


In [ ]:
results_v2 = []

for sample in eval_samples_v2:
    ft_answer = generate_answer(ft_model, ft_tokenizer, sample["context"], sample["question"])
    base_answer = generate_answer(base_model, base_tokenizer, sample["context"], sample["question"])

    ft_score = keyword_overlap_score(ft_answer, sample["reference_answer"])
    base_score = keyword_overlap_score(base_answer, sample["reference_answer"])

    results_v2.append({
        "question": sample["question"],
        "reference": sample["reference_answer"],
        "ft_answer": ft_answer,
        "ft_score": ft_score,
        "base_answer": base_answer,
        "base_score": base_score,
    })

    print(f"\nQ: {sample['question']}")
    print(f"Reference : {sample['reference_answer']}")
    print(f"Fine-tuned ({ft_score:.2f}): {ft_answer}")
    print(f"Base       ({base_score:.2f}): {base_answer}")

In [ ]:
ft_avg_v2 = sum(r["ft_score"] for r in results_v2) / len(results_v2)
base_avg_v2 = sum(r["base_score"] for r in results_v2) / len(results_v2)

print(f"\n{'='*50}")
print(f"[v2 - fuller context] Fine-tuned avg: {ft_avg_v2:.1%}")
print(f"[v2 - fuller context] Base avg:       {base_avg_v2:.1%}")
print(f"Improvement: {(ft_avg_v2 - base_avg_v2):.1%} points")